# Path 3 — Market reactions & event study

Join **macro surprises** with **forward returns** and optional **vol tags**.
Requires: bootstrapped DB, `vol_indices` (bootstrap or `python -m mini_hedge.cli fetch-vol`), and yfinance for SPY/TLT.


In [1]:
import os, sys
from pathlib import Path

_cwd = Path.cwd().resolve()
if (_cwd / "mini_hedge").is_dir():
    project_root = _cwd
elif (_cwd.parent / "mini_hedge").is_dir():
    project_root = _cwd.parent
else:
    _ex = (os.environ.get("MINI_HEDGE_ROOT") or "").strip()
    project_root = Path(_ex).resolve() if _ex else None
    if project_root is None or not (project_root / "mini_hedge").is_dir():
        raise RuntimeError("Set cwd to repo root or notebooks/, or MINI_HEDGE_ROOT")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
from mini_hedge.surprises import compute_surprises
from mini_hedge.prices import fetch_yfinance, CPI_RELEASE_DATES
from mini_hedge.event_study import forward_close_returns, tag_events_with_vol

print("OK — imports")

OK — imports


In [2]:
# Build a minimal CPI event table (release dates × surprise z where available)
surp = compute_surprises(method="ema")
releases = pd.DataFrame({"event_date": pd.to_datetime(CPI_RELEASE_DATES)})
# nearest prior month surprise row per release (simplified join on calendar month)
surp["ym"] = surp["date"].dt.to_period("M")
releases["ym"] = (releases["event_date"] - pd.DateOffset(months=1)).dt.to_period("M")
ev = releases.merge(surp[["ym", "surprise", "surprise_zscore", "signal"]], on="ym", how="left")
ev = ev.drop(columns=["ym"]).dropna(subset=["surprise"])
ev.tail(5)

,event_date,surprise,surprise_zscore,signal
167,2025-12-10,-0.464143,-1.333084,-1.0
168,2026-01-15,-0.044307,-0.125022,0.0
169,2026-02-12,0.368518,1.048737,1.0
170,2026-03-11,0.285894,0.817224,1.0
171,2026-04-10,0.720628,1.993131,1.0


In [4]:
spy = fetch_yfinance("SPY", start="1993-01-29")
tlt = fetch_yfinance("TLT", start="2002-07-26")
ret = forward_close_returns(spy, ev.tail(80), event_date_col="event_date", horizons=(0, 1, 5))
ret[["event_date", "surprise_zscore", "ret_h0", "ret_h1", "ret_h5"]].describe()

,event_date,surprise_zscore,ret_h0,ret_h1,ret_h5
count,80,80.000000,80.000000,80.000000,80.000000
mean,2022-11-28 14:42:00,0.024058,-0.001553,-0.079342,-0.009472
min,2019-08-13 00:00:00,-3.433029,-4.874900,-13.976200,-16.788000
25%,2021-04-04 12:00:00,-0.588599,-0.470000,-0.509500,-1.320150
50%,2022-11-26 12:00:00,-0.055669,0.083750,0.266200,0.592300
75%,2024-07-19 12:00:00,0.831878,0.661100,1.018975,1.736400
max,2026-04-10 00:00:00,2.473622,5.495400,6.516500,5.375100
std,NaN,1.064787,1.448353,2.401476,3.206430


In [5]:
tagged = tag_events_with_vol(ev.tail(40), spy_prices=spy, tlt_prices=tlt)
tagged[["event_date", "vix_pre", "move_pre", "rv_spy_h0_h5_ann", "vol_miss_spy"]].tail(8)

,event_date,vix_pre,move_pre,rv_spy_h0_h5_ann,vol_miss_spy
163,2025-08-12,16.250000,81.760002,5.191834,-11.058166
164,2025-09-10,15.040000,82.199997,6.993864,-8.046136
165,2025-10-15,20.809999,79.949997,11.498012,-9.311987
167,2025-12-10,16.930000,75.379997,9.431152,-7.498848
168,2026-01-15,16.750000,58.110001,19.096427,2.346427
169,2026-02-12,17.650000,61.349998,6.091174,-11.558825
170,2026-03-11,24.930000,76.330002,17.280297,-7.649704
171,2026-04-10,19.490000,74.010002,6.314528,-13.175472
